# Qualidade e preparação dos dados

## Objetivo

Avaliar a qualidade dos dados fornecidos, identificar inconsistências e preparar uma base confiável para análise.

## Estrutura do notebook

1. Importação das bibliotecas
2. Leitura dos dados da camada Bronze
3. Conhecimento inicial das bases
4. Identificação de inconsistências
5. Tratamento dos dados
6. Validação e salvamento na camada Silver

In [9]:
import pandas as pd

ativos = pd.read_csv('../data/bronze/ativos.csv', sep=';')
vulnerabilidades = pd.read_csv('../data/bronze/vulnerabilidades.csv', sep=';')


## 1. Análise inicial das bases

In [45]:
ativos.head()

,ativo_id,sistema,criticidade,dominio,ambiente,localidade,status_ativo
0,A0001,Sistema_1,Crítica,Dominio_2,Homologação,São Paulo,Em desativação
1,A0002,Sistema_2,Média,Dominio_3,Produção,São Paulo,Em desativação
2,A0003,Sistema_3,Média,Dominio_4,Homologação,São Paulo,Ativo
3,A0004,Sistema_4,Alta,Dominio_5,Homologação,Campinas,Ativo
4,A0005,Sistema_5,Alta,Dominio_6,Produção,Recife,Ativo


In [12]:
ativos.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ativo_id      200 non-null    str  
 1   sistema       200 non-null    str  
 2   criticidade   199 non-null    str  
 3   dominio       200 non-null    str  
 4   ambiente      198 non-null    str  
 5   localidade    200 non-null    str  
 6   status_ativo  200 non-null    str  
dtypes: str(7)
memory usage: 11.1 KB


In [68]:
vulnerabilidades.head()

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
0,V00001,A0078,Baixa,Cloud Security,21/03/2026,NaN,Em tratamento,2.8
1,V00002,A0060,Alta,Qualys,01/06/2026,NaN,Em tratamento,7.5
2,V00003,A0080,Alta,Veracode,18/08/2026,NaN,Aberta,1.7
3,V00004,A0086,Alta,Qualys,16/06/2026,NaN,Aberta,2.5
4,V00005,A0089,Alta,Cloud Security,12/02/2026,NaN,Em tratamento,4.4


In [13]:
vulnerabilidades.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   vuln_id        2000 non-null   str    
 1   ativo_id       2000 non-null   str    
 2   severidade     1974 non-null   str    
 3   origem         2000 non-null   str    
 4   data_abertura  2000 non-null   str    
 5   data_correcao  469 non-null    str    
 6   status         2000 non-null   str    
 7   cvss_score     2000 non-null   float64
dtypes: float64(1), str(7)
memory usage: 125.1 KB


## 2. Identificação de valores ausentes

In [43]:
print(f'Valores ausentes em ativos: Total {ativos.isna().sum().sum()}')
display(ativos.isna().sum())

print('-' * 50)

print(f'Valores ausentes em vulnerabilidades: Total {vulnerabilidades.isna().sum().sum()}')
display(vulnerabilidades.isna().sum())


Valores ausentes em ativos: Total 3


ativo_id        0
sistema         0
criticidade     1
dominio         0
ambiente        2
localidade      0
status_ativo    0
dtype: int64

--------------------------------------------------
Valores ausentes em vulnerabilidades: Total 1557


vuln_id             0
ativo_id            0
severidade         26
origem              0
data_abertura       0
data_correcao    1531
status              0
cvss_score          0
dtype: int64

### Validação entre status e data de correção

In [ ]:
vulnerabilidades_status_corrigida = vulnerabilidades[vulnerabilidades['status'] == 'Corrigida']

vulnerabilidades_status_corrigida_na = vulnerabilidades_status_corrigida[vulnerabilidades_status_corrigida['data_correcao'].isna()]
vulnerabilidades_status_corrigida_na

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score


In [69]:
vulnerabilidades_status_nao_corrigida = vulnerabilidades[vulnerabilidades['status'] != 'Corrigida']

vulnerabilidades_status_nao_corrigida_not_na = vulnerabilidades_status_nao_corrigida[vulnerabilidades_status_nao_corrigida['data_correcao'].notna()]
vulnerabilidades_status_nao_corrigida_not_na

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score


**Conclusão:** Não foram encontradas vulnerabilidades corrigidas sem data de correção, nem vulnerabilidades não corrigidas com data de correção. Portanto, a ausência de `data_correcao` é esperada para os demais status.

### Validação do formato de datas

In [76]:
data_validada = pd.to_datetime(vulnerabilidades['data_abertura'], format='%d/%m/%Y', errors='coerce')
data_invalida = vulnerabilidades[data_validada.isna()]
print(f'Total de datas inválidas: {data_invalida.shape[0]}')
data_invalida.head()

Total de datas inválidas: 78


,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
46,V00047,A0167,Alta,Pentest,03-29-2026,NaN,Em tratamento,2.6
70,V00071,A0200,Alta,Qualys,2026-13-01,31/08/2026,Corrigida,4.8
88,V00089,A0123,Média,Cloud Security,31/02/2026,NaN,Aceita,1.0
93,V00094,A0150,Baixa,Veracode,06-16-2026,NaN,Aberta,3.8
140,V00141,A0138,Crítica,Cloud Security,05-30-2026,17/06/2026,Corrigida,2.8


Foram identificadas 78 datas de abertura inválidas. Há registros em formatos diferentes do esperado, como `mm-dd-aaaa` e `aaaa-dd-mm`, além de datas inexistentes, como `31/02/2026`.